# 08. 예외 처리 & context manager (with)

안정적인 파이프라인 코드를 위한 예외 처리 패턴과, 리소스를 안전하게 열고 닫는 `with` 문.

## try / except / else / finally

In [1]:
def safe_divide(a, b):
    try:
        result = a / b
    except ZeroDivisionError as e:
        print(f"0으로 나눌 수 없음: {e}")
        return None
    except TypeError as e:
        print(f"타입이 잘못됨: {e}")
        return None
    else:
        print("정상적으로 나눗셈 성공")  # 예외가 없을 때만 실행
        return result
    finally:
        print("항상 실행됨 (로그 등 마무리 작업)")

print(safe_divide(10, 2))
print(safe_divide(10, 0))


정상적으로 나눗셈 성공
항상 실행됨 (로그 등 마무리 작업)
5.0
0으로 나눌 수 없음: division by zero
항상 실행됨 (로그 등 마무리 작업)
None


## 커스텀 예외와 예외 계층

내가 만든 라이브러리/모듈이면 `Exception`을 상속한 전용 예외를 만들어 쓰는 게 좋습니다 — 호출하는 쪽에서 `except MyError`로 세밀하게 잡을 수 있습니다.

In [2]:
class ModelError(Exception):
    """모델 관련 에러의 베이스 클래스"""
    pass

class ModelNotLoadedError(ModelError):
    pass

class InvalidInputShapeError(ModelError):
    def __init__(self, expected, got):
        super().__init__(f"shape 불일치: expected={expected}, got={got}")
        self.expected = expected
        self.got = got

def predict(model_loaded, input_shape):
    if not model_loaded:
        raise ModelNotLoadedError("먼저 model.load()를 호출하세요")
    if input_shape != (224, 224, 3):
        raise InvalidInputShapeError((224, 224, 3), input_shape)
    return "예측 완료"

try:
    predict(True, (128, 128, 3))
except ModelError as e:  # 공통 베이스로 한 번에 잡기
    print(f"[{type(e).__name__}] {e}")


[InvalidInputShapeError] shape 불일치: expected=(224, 224, 3), got=(128, 128, 3)


## with 문: context manager

파일, DB 커넥션, 락(lock) 등 '열었으면 반드시 닫아야 하는' 리소스를 다룰 때 씁니다. 예외가 나도 `__exit__`이 호출되어 정리가 보장됩니다.

In [3]:
import tempfile, os

path = os.path.join(tempfile.gettempdir(), "demo.txt")

with open(path, "w") as f:
    f.write("hello")
# 블록을 벗어나면 f.close()가 자동으로 호출됨 (예외가 나도!)
print("닫혔는가:", f.closed)

os.remove(path)


닫혔는가: True


## 직접 context manager 만들기 (클래스 방식: `__enter__` / `__exit__`)

In [4]:
import time

class Timer:
    def __enter__(self):
        self.start = time.perf_counter()
        return self  # as 뒤에 바인딩될 객체

    def __exit__(self, exc_type, exc_value, traceback):
        elapsed = time.perf_counter() - self.start
        print(f"elapsed: {elapsed*1000:.2f}ms")
        return False  # False -> 발생한 예외를 그대로 전파 (삼키지 않음)

with Timer() as t:
    total = sum(range(1_000_000))
print(total)


elapsed: 12.25ms
499999500000


## 직접 context manager 만들기 (함수 방식: contextlib)

클래스 없이, 제너레이터 함수 + `@contextmanager`로 더 짧게 만들 수 있습니다. `yield` 이전이 `__enter__`, 이후가 `__exit__` 역할을 합니다.

In [5]:
from contextlib import contextmanager

@contextmanager
def timer(label):
    import time
    start = time.perf_counter()
    try:
        yield  # 이 지점에서 with 블록 내부 코드가 실행됨
    finally:
        print(f"[{label}] {(time.perf_counter() - start)*1000:.2f}ms")

with timer("데이터 로딩"):
    data = list(range(500_000))


[데이터 로딩] 10.06ms


## 헷갈렸던 점 / 더 찾아볼 것

- `except Exception:`처럼 너무 넓게 잡으면 디버깅이 어려워짐 — 가능하면 구체적인 예외 타입을 잡을 것
- `__exit__`에서 `True`를 반환하면 예외를 '삼켜버림' — 의도치 않게 에러를 숨기지 않도록 주의